# 🎬 Movie Recommendation System (Content-Based Filtering)

Built for **Google Colab**. Run the cells from top to bottom (**Runtime → Run all**).

**What you'll build:** type a movie you like → get 5 similar movies, based on the similarity of their genres (TF-IDF + cosine similarity).

**Steps in this notebook**
1. Set up the project and download MovieLens
2. Explore the dataset
3. Understand the algorithm (TF-IDF + cosine similarity)
4. Build the recommendation engine
5. Test the system
6. Upgrade: genre filter, explanations, packaging into `src/`
7. *(Optional)* Collaborative filtering with `ratings.csv`

Libraries: Python, Pandas, scikit-learn (all preinstalled in Colab).

## Step 1: Set up your project
Create the folders and download the small MovieLens dataset from GroupLens.

In [ ]:
!mkdir -p data src notebooks
!wget -q -O data/ml-latest-small.zip https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -q -o data/ml-latest-small.zip -d data
!ls data/ml-latest-small

## Step 2: Explore the dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

movies = pd.read_csv("data/ml-latest-small/movies.csv")
print(movies.head())
print("\nShape:", movies.shape)
print("\nMissing values:\n", movies.isnull().sum())

**Columns:** `movieId` (unique ID), `title` (title + release year), `genres` (genres separated by `|`).

Now check duplicates, movies with no genres, and how genres are distributed.

In [ ]:
print("Duplicate movie IDs:", movies["movieId"].duplicated().sum())
print("Duplicate titles:   ", movies["title"].duplicated().sum())
print("No genres listed:   ", (movies["genres"] == "(no genres listed)").sum())

genre_counts = movies["genres"].str.split("|").explode().value_counts()
print("\nMovies per genre:\n", genre_counts)

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
genre_counts.drop("(no genres listed)", errors="ignore").plot.barh(ax=ax[0], title="Movies per genre")
movies["genres"].str.count(r"\|").add(1).value_counts().sort_index().plot.bar(ax=ax[1], title="Genres per movie")
plt.tight_layout(); plt.show()

## Step 3: Understand the recommendation algorithm

- **TF-IDF** turns text into numbers, giving more weight to terms that are informative (rare across movies) and less to very common ones.
- **Cosine similarity** measures how similar two vectors are (1 = same direction, 0 = nothing in common).
- **Recommendation:** take the selected movie's similarity to every other movie, sort high → low, return the top 5.

Here is a tiny example with the three movies from the roadmap:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

toy = pd.DataFrame({
    "title":  ["Toy Story", "Jumanji", "Heat"],
    "genres": ["Adventure|Animation|Children|Comedy", "Adventure|Children|Fantasy", "Action|Crime|Thriller"],
})
toy["genre_text"] = toy["genres"].str.replace("|", " ", regex=False)   # pipes -> spaces

vec = TfidfVectorizer(token_pattern=r"[^\s]+")
X = vec.fit_transform(toy["genre_text"])
print(pd.DataFrame(X.toarray(), index=toy["title"], columns=vec.get_feature_names_out()).round(2))
print()
print(pd.DataFrame(cosine_similarity(X), index=toy["title"], columns=toy["title"]).round(2))

Toy Story and Jumanji share *Adventure* and *Children*, so they're similar. Heat shares nothing with either (score 0).

## Step 4: Build the recommendation engine

First, the steps by hand on the real dataset:

In [ ]:
# 1. Replace pipes with spaces
genre_text = movies["genres"].str.replace("|", " ", regex=False).replace("(no genres listed)", "")
# 2. TF-IDF vectors
tfidf = TfidfVectorizer(token_pattern=r"[^\s]+").fit_transform(genre_text)
print("TF-IDF matrix:", tfidf.shape)
# 3-5. similarity of one movie to all others, then sort
idx = movies.index[movies["title"] == "Toy Story (1995)"][0]
scores = cosine_similarity(tfidf[idx], tfidf).ravel()
scores[idx] = -1                                   # 7. exclude the movie itself
top = scores.argsort()[::-1][:5]
movies.loc[top, ["title", "genres"]].assign(similarity=scores[top].round(2))

That works, but it's not reusable and doesn't handle bad input. Below we package it into two modules (`src/preprocessing.py` and `src/recommender.py`) that:
- clean the data (duplicates, missing genres, release year),
- build the TF-IDF + pairwise cosine similarity matrix,
- match titles flexibly (`"Toy Story"`, `"toy story (1995)"`, `"The Matrix"` → `"Matrix, The (1999)"`),
- handle unknown titles (with "did you mean…" suggestions) and movies with no genres,
- never return the selected movie, and explain results via shared genres.

The `%%writefile` cells save the files into `src/` so you can download them for GitHub later.

In [ ]:
%%writefile src/preprocessing.py
"""Loading, inspecting and cleaning the MovieLens movies table."""
from pathlib import Path

import pandas as pd

NO_GENRES = "(no genres listed)"
DEFAULT_PATH = Path("data/ml-latest-small/movies.csv")


def load_movies(path=DEFAULT_PATH) -> pd.DataFrame:
    """Read movies.csv (columns: movieId, title, genres)."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Download the MovieLens Latest Small dataset "
            "from https://grouplens.org/datasets/movielens/latest/ and extract it "
            "into the data/ folder (see data/README.md)."
        )
    return pd.read_csv(path)


def data_quality_report(movies: pd.DataFrame) -> dict:
    """Quick checks: size, missing values, duplicates, movies without genres."""
    return {
        "rows": len(movies),
        "missing_values": movies.isnull().sum().to_dict(),
        "duplicate_movie_ids": int(movies["movieId"].duplicated().sum()),
        "duplicate_titles": int(movies["title"].duplicated().sum()),
        "movies_without_genres": int((movies["genres"] == NO_GENRES).sum()),
    }


def clean_movies(movies: pd.DataFrame) -> pd.DataFrame:
    """Return a cleaned copy with helper columns used by the recommender.

    Added columns:
      genre_list : list of genres (empty list when none are listed)
      genre_text : genres separated by spaces (the text fed to TF-IDF)
      year       : release year extracted from the title (may be missing)
    """
    df = movies.drop_duplicates(subset="movieId").dropna(subset=["title"]).copy()
    df["title"] = df["title"].str.strip()
    df["genres"] = df["genres"].fillna(NO_GENRES)

    df["genre_list"] = df["genres"].apply(
        lambda g: [] if g == NO_GENRES else g.split("|")
    )
    # Step 1 of the roadmap: replace the pipe characters with spaces.
    df["genre_text"] = df["genre_list"].apply(" ".join)
    df["year"] = df["title"].str.extract(r"\((\d{4})\)\s*$")[0].astype("Int64")
    return df.reset_index(drop=True)

In [ ]:
%%writefile src/recommender.py
"""Content-based movie recommender (TF-IDF on genres + cosine similarity)."""
import difflib
import re

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from .preprocessing import DEFAULT_PATH, clean_movies, load_movies

_YEAR = re.compile(r"\s*\(\d{4}\)\s*$")
_ARTICLE = re.compile(r"^(the|a|an)\s+(.+)$")


class MovieNotFoundError(ValueError):
    """The requested title is not in the dataset."""


class NoGenreInfoError(ValueError):
    """The movie has no genre information, so nothing can be compared."""


class MovieRecommender:
    def __init__(self, movies: pd.DataFrame, chunk_size: int = 1000):
        """`movies` must be the output of preprocessing.clean_movies()."""
        self.movies = movies.reset_index(drop=True)

        # Convert the genre strings into TF-IDF vectors.
        self.vectorizer = TfidfVectorizer(token_pattern=r"[^\s]+")
        self.tfidf = self.vectorizer.fit_transform(self.movies["genre_text"])

        # Pairwise cosine similarity matrix (float32, computed in chunks to save memory).
        self.similarity = self._pairwise_similarity(chunk_size)

        # Lookup helpers for matching user-typed titles to rows.
        self._title_lower = self.movies["title"].str.lower()
        self._title_no_year = self._title_lower.str.replace(_YEAR, "", regex=True)

    # ------------------------------------------------------------------ setup
    @classmethod
    def from_csv(cls, path=DEFAULT_PATH):
        return cls(clean_movies(load_movies(path)))

    def _pairwise_similarity(self, chunk_size: int) -> np.ndarray:
        n = self.tfidf.shape[0]
        sim = np.empty((n, n), dtype=np.float32)
        for start in range(0, n, chunk_size):
            stop = start + chunk_size
            sim[start:stop] = cosine_similarity(self.tfidf[start:stop], self.tfidf)
        return sim

    # --------------------------------------------------------- title matching
    def find_movie(self, title: str):
        """Return the row index for a title, or None.

        Tries: exact title, title without year, MovieLens-style articles
        ("The Matrix" -> "Matrix, The"), then a partial match.
        """
        q = title.strip().lower()
        if not q:
            return None
        variants = [q]
        m = _ARTICLE.match(q)
        if m:
            variants.append(f"{m.group(2)}, {m.group(1)}")

        for series in (self._title_lower, self._title_no_year):
            for v in variants:
                hits = self.movies.index[series == v]
                if len(hits):
                    return int(hits[0])
        for v in variants:
            hits = self.movies.index[self._title_lower.str.contains(v, regex=False)]
            if len(hits):
                return int(hits[0])
        return None

    def suggest_titles(self, query: str, n: int = 5) -> list:
        """Closest titles to a (possibly misspelled) query."""
        names = self._title_no_year.tolist()
        close = difflib.get_close_matches(query.strip().lower(), names, n=n, cutoff=0.6)
        lookup = dict(zip(reversed(names), reversed(self.movies["title"].tolist())))
        return [lookup[c] for c in close]

    def genres(self) -> list:
        return sorted({g for lst in self.movies["genre_list"] for g in lst})

    # --------------------------------------------------------- recommendations
    def recommend_movies(self, title: str, n: int = 5, genre: str = None) -> pd.DataFrame:
        """Return the top-n most similar movies as a DataFrame.

        Columns: title, genres, similarity, shared_genres.
        The selected movie is never included. `genre` optionally restricts the results.
        Raises MovieNotFoundError / NoGenreInfoError / ValueError with clear messages.
        """
        idx = self.find_movie(title)
        if idx is None:
            hint = self.suggest_titles(title)
            msg = f"Movie '{title}' was not found."
            if hint:
                msg += " Did you mean: " + "; ".join(hint) + "?"
            raise MovieNotFoundError(msg)

        selected = self.movies.loc[idx]
        if not selected["genre_list"]:
            raise NoGenreInfoError(
                f"'{selected['title']}' has no genre information, so it can't be compared."
            )

        sims = self.similarity[idx].astype(float).copy()
        sims[idx] = -1.0  # exclude the selected movie itself

        if genre is not None:
            wanted = {g.lower(): g for g in self.genres()}.get(genre.strip().lower())
            if wanted is None:
                raise ValueError(f"Unknown genre '{genre}'. Choose from: {', '.join(self.genres())}")
            has_genre = self.movies["genre_list"].apply(lambda g: wanted in g).to_numpy()
            sims[~has_genre] = -1.0

        order = np.argsort(-sims, kind="stable")  # stable => deterministic ties
        order = [i for i in order if sims[i] > 0][:n]

        result = self.movies.loc[order, ["title", "genres"]].copy()
        result["similarity"] = sims[order].round(3)
        result["shared_genres"] = [
            ", ".join(g for g in selected["genre_list"] if g in self.movies.at[i, "genre_list"])
            for i in order
        ]
        return result.reset_index(drop=True)


def print_recommendations(rec: MovieRecommender, title: str, n: int = 5, genre: str = None):
    """Friendly console output; catches the 'expected' errors."""
    try:
        out = rec.recommend_movies(title, n=n, genre=genre)
    except ValueError as err:
        print(err)
        return
    shown = rec.movies.loc[rec.find_movie(title), "title"]
    suffix = f" (genre filter: {genre})" if genre else ""
    print(f"Because you picked '{shown}'{suffix}:")
    if out.empty:
        print("  No similar movies found.")
    for i, row in out.iterrows():
        print(f"{i + 1}. {row['title']}  [{row['genres']}]")
        print(f"   similarity {row['similarity']:.2f} | shared genres: {row['shared_genres']}")


def main():
    """Simple command-line menu:  python -m src.recommender"""
    rec = MovieRecommender.from_csv()
    menu = "\n1) Recommend  2) Recommend with genre filter  3) List genres  4) Quit"
    while True:
        print(menu)
        choice = input("Choose: ").strip()
        if choice == "1":
            print_recommendations(rec, input("Movie title: "))
        elif choice == "2":
            print_recommendations(rec, input("Movie title: "), genre=input("Genre: "))
        elif choice == "3":
            print(", ".join(rec.genres()))
        elif choice == "4":
            break
        else:
            print("Please enter 1-4.")


if __name__ == "__main__":
    main()

In [ ]:
!touch src/__init__.py
from src.preprocessing import load_movies, data_quality_report, clean_movies
from src.recommender import MovieRecommender, print_recommendations

raw = load_movies()
print(data_quality_report(raw))
recommender = MovieRecommender(clean_movies(raw))
print("Similarity matrix:", recommender.similarity.shape)

## Step 5: Test your system
Try at least five different movies, and check the results look right.

In [ ]:
for t in ["Toy Story", "Heat", "Interstellar", "Jumanji", "The Matrix"]:
    print_recommendations(recommender, t)
    print()

**Checks:** is the selected movie excluded? Do the results share genres? What about bad input?

In [ ]:
# 1. Selected movie is never in its own results
for t in ["Toy Story (1995)", "Heat (1995)", "Interstellar (2014)", "Jumanji (1995)"]:
    out = recommender.recommend_movies(t, n=10)
    assert t not in out["title"].values
print("✓ selected movie excluded")

# 2. Results are sorted by similarity
assert recommender.recommend_movies("Toy Story")["similarity"].is_monotonic_decreasing
print("✓ sorted from most to least similar\n")

# 3. Unknown / misspelled titles
print_recommendations(recommender, "Toy Stroy")
print_recommendations(recommender, "zzzz not a movie")
print()

# 4. A movie with no genre information
no_genre_title = recommender.movies.loc[recommender.movies["genre_list"].str.len() == 0, "title"].iloc[0]
print_recommendations(recommender, no_genre_title)

## Step 6: Upgrade it

Genre filter and explanations (shared genres). `recommend_movies()` **returns** a DataFrame instead of just printing:

In [ ]:
print_recommendations(recommender, "Toy Story", n=5, genre="Comedy")
print()
recommender.recommend_movies("Interstellar", n=5, genre="Drama")

### Put it on GitHub
1. In Colab: **File → Download .ipynb** (save into `notebooks/`) and download the two files from the **Files** panel (📁): `src/preprocessing.py`, `src/recommender.py`.
2. Add `app.py`, `requirements.txt`, `.gitignore` and `data/README.md` from the project ZIP, then write your own `README.md` on GitHub.
3. **Don't upload the dataset** (see `data/README.md`).


## Step 7 (Optional): Collaborative filtering with `ratings.csv`

Instead of *"which movies have similar genres?"* this asks *"which movies do users with similar rating patterns tend to like?"*.
We use **item-based** collaborative filtering: two movies are similar if the same users rate them similarly (after subtracting each user's average, so generous vs. harsh raters are comparable).

In [ ]:
ratings = pd.read_csv("data/ml-latest-small/ratings.csv")

# Keep movies with enough ratings to be reliable
n_ratings = ratings.groupby("movieId").size()
ratings = ratings[ratings["movieId"].isin(n_ratings[n_ratings >= 50].index)]

user_item = ratings.pivot_table(index="userId", columns="movieId", values="rating")
centered = user_item.sub(user_item.mean(axis=1), axis=0).fillna(0)
item_sim = pd.DataFrame(cosine_similarity(centered.T), index=user_item.columns, columns=user_item.columns)

def collab_recommend(title, n=5):
    idx = recommender.find_movie(title)
    if idx is None:
        raise ValueError(f"Movie '{title}' was not found.")
    movie_id = recommender.movies.at[idx, "movieId"]
    if movie_id not in item_sim.index:
        raise ValueError("Not enough ratings for this movie.")
    s = item_sim[movie_id].drop(movie_id).nlargest(n)
    out = recommender.movies.set_index("movieId").loc[s.index, ["title", "genres"]]
    out["similarity"] = s.round(3).values
    return out.reset_index(drop=True)

print("Collaborative filtering:")
display(collab_recommend("Toy Story"))
print("Content-based (genres only):")
display(recommender.recommend_movies("Toy Story")[["title", "genres", "similarity"]])

Compare the two lists: collaborative filtering can surface movies with *different* genres that the same audience likes, while content-based only sees genres. A natural next step is a **hybrid** of both.

**Limitations:** the content model only uses genres (no plot, cast or tags), so many movies tie; it doesn't use any user's history.